In [ ]:
# fr/python-101/hard/02-exploring-corpus
# Generated companion notebook for the PyDA course.
# Run cells top-to-bottom (or in any order) to follow the lesson.

print("PyDA — ready 🚀")


In [ ]:
# 💾 Load the course datasets into this environment
# The course data files live in the PyDA repo; pull them so
# `open("…")` / `pd.read_csv("…")` work exactly like on disk.
import os
def _fetch(name, aliases=()):
    if os.path.exists(name):
        return
    url = f"https://raw.githubusercontent.com/abderrahim-lectures/python-data-analysis-course/main/public/datasets/{name}"
    os.system(f"curl -sL -o {name} {url}")
    for alias in aliases:
        if not os.path.exists(alias):
            os.system(f"cp {name} {alias}")

_fetch("slm-corpus.csv", ())


Explorez avant de traiter

Charger des données est l'étape un. L'étape deux consiste à comprendre ce que vous avez chargé. Un corpus peut avoir des valeurs manquantes, des lignes en double, des caractères encodés qui ressemblent à du charabia, ou du texte trop court pour être utile. Passer cinq minutes à explorer maintenant vous fait gagner des heures de débogage plus tard.

## Concepts clés

### Compter les lignes et les colonnes

Les statistiques les plus simples vous en disent beaucoup. Un corpus de 5 lignes ne produira pas un modèle utile ; un de 50 000 lignes pourrait nécessiter un chargement en blocs :


In [ ]:
import csv

with open("slm-corpus.csv", newline="") as f:
    reader = csv.DictReader(f)
    rows = list(reader)

print(f"Rows:    {len(rows)}")
print(f"Columns: {list(rows[0].keys())}")


### Mesurer la longueur du texte

Les modèles de langue ont besoin de suffisamment de texte pour apprendre des motifs. Vérifiez le nombre total de caractères et la longueur moyenne des lignes :


In [ ]:
total_chars = sum(len(row["text"]) for row in rows)
avg_len = total_chars / len(rows) if rows else 0

print(f"Total characters: {total_chars:,}")
print(f"Average row length: {avg_len:.0f} characters")


Un corpus avec une moyenne de 10 caractères par ligne est trop court — le modèle n'aura pas assez de contexte pour apprendre les séquences de mots.

### Prévisualiser le texte d'échantillon

Lisez quelques lignes pour vous faire une idée du contenu. Quelle langue est-ce ? Quels sujets couvre-t-il ? Le texte est-il propre ou bruité ?


In [ ]:
for i, row in enumerate(rows[:5]):
    preview = row["text"][:150].replace("\n", " ")
    print(f"[{i}] {preview}...")


### Trouver les doublons

Les lignes en double gonflent les comptes de mots sans ajouter de nouvelles informations. Détectez-les en convertissant les lignes en un ensemble :


In [ ]:
unique_texts = set(row["text"] for row in rows)
print(f"Unique rows: {len(unique_texts)} / {len(rows)}")

if len(unique_texts) < len(rows):
    print(f"Warning: {len(rows) - len(unique_texts)} duplicate rows found")


### Vérifier les lignes vides ou courtes

Les lignes vides ou très courtes n'apporteront pas de bigrammes utiles. Filtrez-les :


In [ ]:
short_rows = [row for row in rows if len(row["text"].split()) < 3]
print(f"Rows with fewer than 3 words: {len(short_rows)}")


Une fonction de résumé de corpus combine toutes ces vérifications :


In [ ]:
def corpus_summary(path):
    import csv
    with open(path, newline="") as f:
        reader = csv.DictReader(f)
        rows = list(reader)

    texts = [row["text"] for row in rows]
    total_chars = sum(len(t) for t in texts)
    unique = len(set(texts))

    print(f"Rows: {len(rows)}")
    print(f"Unique: {unique}")
    print(f"Total chars: {total_chars:,}")
    print(f"Avg length: {total_chars / len(rows):.0f}")
    print(f"Columns: {list(rows[0].keys())}")


## Essayez

Exécutez `corpus_summary("slm-corpus.csv")` et notez :
1. Combien de lignes le corpus contient-il ?
2. Y a-t-il des doublons ?
3. La longueur moyenne du texte est-elle suffisante pour construire des bigrammes significatifs (au moins 20+ mots par ligne) ?

## Points clés

- Explorez toujours vos données avant de les traiter — vérifiez les comptes, les longueurs et les doublons
- Les lignes courtes ou vides ajoutent du bruit ; filtrez-les sur la base d'un nombre minimum de mots
- Les lignes en double gonflent les comptes de fréquence sans ajouter de nouveaux motifs
- Une fonction de résumé rapide fait gagner du temps entre projets

## Défi pratique

Écrivez une fonction `corpus_quality(path)` qui charge un CSV et renvoie un dict avec ces clés : `"rows"`, `"unique"`, `"total_chars"`, `"avg_length"`, `"min_length"`, `"max_length"`. Utilisez-la pour évaluer si `slm-corpus.csv` convient à la modélisation par bigrammes.


In [ ]:
def corpus_quality(path):
    import csv
    with open(path, newline="") as f:
        reader = csv.DictReader(f)
        rows = list(reader)

    texts = [row["text"] for row in rows]
    lengths = [len(t.split()) for t in texts]

    return {
        "rows": len(rows),
        "unique": len(set(texts)),
        "total_chars": sum(len(t) for t in texts),
        "avg_length": sum(lengths) / len(lengths) if lengths else 0,
        "min_length": min(lengths) if lengths else 0,
        "max_length": max(lengths) if lengths else 0,
    }


In [ ]:
# The end. Practice on your own — each cell is a minimal, runnable chunk.
